In [26]:
import pandas as pd
from  sklearn.neighbors import KNeighborsRegressor,KNeighborsClassifier
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.pipeline  import Pipeline
from sklearn.metrics import classification_report

In [27]:
df = pd.read_csv("loan_approval_dataset.csv")
df

,Age,Salary,Credit_Score,Loan_Amount,Loan_Term,Employment_Status,Residence_Type,Previous_Default,Loan_Approved
0,56,136748,584,38209,36 months,Employed,Owned,Yes,Yes
1,46,25287,815,27424,24 months,Self-Employed,Rented,No,Yes
2,32,146593,398,42396,12 months,Unemployed,Rented,Yes,Yes
3,60,54387,696,11370,24 months,Unemployed,Owned,No,No
4,25,28512,788,14528,12 months,Employed,Owned,No,No
...,...,...,...,...,...,...,...,...,...
995,22,49241,500,41020,24 months,Self-Employed,Owned,No,Yes
996,40,116214,423,12415,48 months,Self-Employed,Owned,No,Yes
997,27,64569,300,28155,36 months,Self-Employed,Rented,Yes,Yes
998,61,31745,490,48884,12 months,Self-Employed,Mortgage,No,Yes


In [28]:
x = df.drop(columns = "Loan_Approved")
y = df.Loan_Approved

In [29]:
xtrain, xtest, ytrain, ytest = train_test_split(x,y,random_state = 42,train_size = 0.8)

In [30]:
num_cols = x.select_dtypes(include="number").columns
cat_cols = x.select_dtypes(include="object").columns

In [31]:
cat_cols.nunique()

4

In [32]:
preprocessing = ColumnTransformer(
    transformers=[
        ("scaler",StandardScaler(),num_cols),
        ("encoder",OneHotEncoder(),cat_cols)
    ]
)

In [33]:
main_pipe =Pipeline(
    steps=[
        ("pre",preprocessing),
        ("model",KNeighborsClassifier())
    ]
)

In [34]:
# main_pipe.fit(xtrain,ytrain)

In [35]:
grid_cv = GridSearchCV(
    estimator=main_pipe,
    param_grid={
        "model__metric":["manhattan","euclidean"],
        "model__n_neighbors":[3,7,25,27,29,38]
    },
    cv = 5,
    n_jobs = -1,
    verbose=1
)
grid_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


c:\Users\bthar\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [    nan     nan     nan     nan     nan     nan 0.51    0.4675  0.48125
 0.48375 0.46875 0.46875]
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__metric': ['manhattan', 'euclidean'], 'model__n_neighbors': [3, 7, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is

In [36]:
results = grid_cv.cv_results_
pd.DataFrame(results)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__metric,param_model__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.014189,0.003560,0.009389,0.001745,manhattan,3,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
1,0.015806,0.002106,0.008910,0.001274,manhattan,7,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
2,0.016564,0.001733,0.010358,0.003426,manhattan,25,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
3,0.016996,0.001627,0.011686,0.001046,manhattan,27,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
4,0.017671,0.001116,0.011373,0.000619,manhattan,29,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
5,0.018012,0.001443,0.010685,0.001047,manhattan,38,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
6,0.017681,0.001121,0.013302,0.000542,euclidean,3,"{'model__metric': 'euclidean', 'model__n_neigh...",0.48750,0.53125,0.51875,0.56250,0.45000,0.51000,0.038446,1
7,0.016171,0.002004,0.092447,0.064471,euclidean,7,"{'model__metric': 'euclidean', 'model__n_neigh...",0.46250,0.46875,0.48125,0.51250,0.41250,0.46750,0.032452,6
8,0.015565,0.001944,0.041005,0.052526,euclidean,25,"{'model__metric': 'euclidean', 'model__n_neigh...",0.42500,0.48125,0.55625,0.49375,0.45000,0.48125,0.044546,3
9,0.015204,0.002228,0.039373,0.047210,euclidean,27,"{'model__metric': 'euclidean', 'model__n_neigh...",0.43750,0.45625,0.58750,0.49375,0.44375,0.48375,0.055424,2


In [37]:
grid_cv.best_params_

{'model__metric': 'euclidean', 'model__n_neighbors': 3}